# **Fase 5: MLOps y Produccion**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 11 febrero, 2026

# Notebook 10: MLflow Tracking

**Objetivo**: Rastrear experimentos, métricas y modelos con MLflow

**Conceptos clave:**
- **Experiment**: Agrupación lógica de runs (un proyecto)
- **Run**: Una ejecución individual (un modelo entrenado)
- **Parameters**: Hiperparámetros registrados (regParam, maxIter, etc.)
- **Metrics**: Métricas de rendimiento (RMSE, R², etc.)
- **Artifacts**: Archivos guardados (modelos, gráficos, etc.)

**Actividades:**
1. Configurar MLflow tracking server
2. Registrar experimentos con hiperparámetros
3. Guardar métricas y artefactos
4. Comparar runs en MLflow UI


## 1. Configuración de SparkSession

Se crea una sesión de spark configurada para ejecutarse en modo local, asignando memoria al driver.

In [11]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
import mlflow
import mlflow.spark
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"


# %%
spark = SparkSession.builder \
    .appName("SECOP_MLflow") \
    .master("local[*]") \
    .getOrCreate()
mlflow.set_tracking_uri("http://mlflow:5000")
spark.sparkContext.setLogLevel("ERROR")

## 2. Reto 1: Configurar MLflow tracking server y experimento

**Objetivo**: Conectarse al tracking server y crear un experimento.
**Conceptos**:
- `mlflow.set_tracking_uri()`: URL del servidor MLflow
- `mlflow.set_experiment()`: Nombre del experimento
- El tracking server almacena todos los runs, métricas y artefactos

 **Instrucciones**:
1. Configura la URI del tracking server
2. Crea o selecciona un experimento con nombre descriptivo

**Pregunta**: ¿Por qué es importante un tracking server centralizado en lugar de guardar métricas en archivos locales?


In [12]:
experiment_name = "/SECOP_Contratos_Prediccion"
mlflow.set_experiment(experiment_name)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento activo: {experiment_name}")

MLflow Tracking URI: http://mlflow:5000
Experimento activo: /SECOP_Contratos_Prediccion


Un tracking server centralizado permite que todos los experimentos, métricas, modelos y artefactos queden almacenados en un solo lugar, facilitando la comparación entre runs y la reproducibilidad de algo que no es posible de forma ordenada con archivos locales.

### 2.1 Cargar datos

In [13]:
df = spark.read.parquet("/opt/spark-data/processed/secop_ml_ready.parquet")
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_pca", "features") \
       .filter(col("label").isNotNull())

train, test = df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train.count():,}")
print(f"Test: {test.count():,}")

# %%
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

Train: 41,925


Test: 10,323


## 3. Reto 2: Registrar experimento baseline con log_param/log_metric

**Objetivo**: Entrenar un modelo sin regularización y registrarlo en MLflow.


In [14]:
from pyspark.ml.regression import LinearRegression

with mlflow.start_run(run_name="baseline_no_regularization"):
    reg_param = 0.0
    elastic_param = 0.0
    max_iter = 100

   
    mlflow.log_param("regParam", reg_param)
    mlflow.log_param("elasticNetParam", elastic_param)
    mlflow.log_param("maxIter", max_iter)
    mlflow.log_param("model_type", "LinearRegression")

    
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        regParam=reg_param,
        elasticNetParam=elastic_param,
        maxIter=max_iter
    )
    model = lr.fit(train)

   
    predictions = model.transform(test)
    rmse = evaluator.evaluate(predictions)

    
    mlflow.log_metric("rmse", rmse)

    
    mlflow.spark.log_model(model, "model")

    print(f"Baseline RMSE: ${rmse:,.2f}")


Baseline RMSE: $5,200,424,547.75


## 4. Reto 3: Registrar multiples modelos (Ridge, Lasso, ElasticNet)

**Objetivo**: Entrenar y registrar varios modelos con diferentes regularizaciones para comparar en MLflow UI.

**Instrucciones**:

Crea al menos 3 runs adicionales:
1. Ridge (L2): regParam=0.1, elasticNetParam=0.0
2. Lasso (L1): regParam=0.1, elasticNetParam=1.0
3. ElasticNet: regParam=0.1, elasticNetParam=0.5

**Cada run debe registrar**:
- Parámetros: regParam, elasticNetParam, maxIter, model_type
- Métricas: rmse, mae, r2
- Artefactos: modelo entrenado

**Pregunta**: ¿Por qué registrar múltiples métricas y no solo RMSE?

Registrar múltiples métricas permite evaluar el modelo desde distintos ángulos: RMSE penaliza errores grandes, MAE es más robusto a outliers y R² indica qué tan bien el modelo explica la variabilidad de los datos.

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="rmse"
)
evaluator_mae = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="mae"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="r2"
)

experiments = [
    {"name": "ridge_l2", "reg": 0.1, "elastic": 0.0, "type": "Ridge"},
    {"name": "lasso_l1", "reg": 0.1, "elastic": 1.0, "type": "Lasso"},
    {"name": "elasticnet", "reg": 0.1, "elastic": 0.5, "type": "ElasticNet"},
]

for exp in experiments:
    with mlflow.start_run(run_name=exp["name"]):
        mlflow.log_param("regParam", exp["reg"])
        mlflow.log_param("elasticNetParam", exp["elastic"])
        mlflow.log_param("maxIter", 100)
        mlflow.log_param("model_type", exp["type"])

        lr = LinearRegression(
            featuresCol="features",
            labelCol="label",
            regParam=exp["reg"],
            elasticNetParam=exp["elastic"],
            maxIter=100
        )

        model = lr.fit(train)
        preds = model.transform(test)

        rmse = evaluator_rmse.evaluate(preds)
        mae = evaluator_mae.evaluate(preds)
        r2 = evaluator_r2.evaluate(preds)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        mlflow.spark.log_model(model, "model")

        print(f"{exp['type']} -> RMSE: ${rmse:,.2f}")


Ridge -> RMSE: $5,200,424,547.74


Lasso -> RMSE: $5,200,424,547.61


ElasticNet -> RMSE: $5,200,424,547.67


Los modelos Ridge, Lasso y ElasticNet presentan valores de RMSE muy similares. Esto indica que, para esta configuración de datos y con un nivel de regularización moderado, el tipo de penalización no tiene un impacto significativo en el desempeño predictivo. Dado que las variables ya fueron transformadas mediante PCA, la multicolinealidad se reduce y los beneficios diferenciales de cada técnica de regularización se atenúan.

## 5. Reto 4: Explorar y comparar runs en MLflow UI

**Objetivo**: Usar la interfaz web de MLflow para comparar experimentos.

**Instrucciones**:

1. Abre MLflow UI en http://localhost:5000
2. Navega al experimento que creaste
3. Compara los runs lado a lado
4. Ordena por RMSE para encontrar el mejor
5. Examina los parámetros y métricas de cada run

**Preguntas**:

- ¿Qué modelo tiene el menor RMSE?
- ¿Hay correlación entre regularización y rendimiento?
- ¿Cómo podrías compartir estos resultados con tu equipo?

**Mejor modelo en MLflow UI:** El modelo con regularización Lasso (L1) presenta el menor RMSE, aunque la diferencia frente a Ridge y ElasticNet es mínima. Esto indica que, para este conjunto de datos y nivel de regularización, Lasso logra un ajuste ligeramente mejor en términos de error cuadrático medio.

**¿Hay correlación entre regularización y rendimiento?** No se observa una correlación fuerte entre el tipo de regularización y el rendimiento del modelo, ya que los valores de RMSE son muy similares entre Ridge, Lasso y ElasticNet. Esto sugiere que, dado que las variables ya fueron transformadas mediante PCA y el regParam es moderado, la regularización no impacta de forma significativa el desempeño predictivo.

**¿Cómo podrías compartir estos resultados con tu equipo?** Los resultados pueden compartirse fácilmente mediante la MLflow UI, permitiendo que el equipo compare los runs, métricas y parámetros de cada modelo en un entorno visual e interactivo.

## 6. Reto 5: Agregar artefactos personalizados (reportes, graficos)

**Objetivo**: Guardar artefactos adicionales (gráficos, reportes) en un run.
**Instrucciones**:
1. Dentro de un run, genera un reporte de métricas en texto
2. Guárdalo como artefacto con `mlflow.log_artifact()`
3. (Bonus) Genera un gráfico de predicciones vs reales y guárdalo

**Funciones útiles**:
- `mlflow.log_artifact(local_path)`: Guarda un archivo
- `mlflow.log_text(text, filename)`: Guarda texto directamente


In [22]:
# %%
# RETO 5: Agregar Artefactos Personalizados (con gráfico profesional)

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import matplotlib.pyplot as plt
import pandas as pd
import tempfile
import os

# Evaluadores
evaluator_rmse = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="rmse"
)
evaluator_mae = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="mae"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="r2"
)

with mlflow.start_run(run_name="best_model_with_artifacts"):

    # Mejores hiperparámetros (ajusta si cambian)
    reg_param = 0.1
    elastic_param = 1.0
    max_iter = 100

    # Log de hiperparámetros
    mlflow.log_param("regParam", reg_param)
    mlflow.log_param("elasticNetParam", elastic_param)
    mlflow.log_param("maxIter", max_iter)
    mlflow.log_param("model_type", "Lasso")

    # Entrenar modelo
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        regParam=reg_param,
        elasticNetParam=elastic_param,
        maxIter=max_iter
    )

    model = lr.fit(train)

    # Predicciones
    predictions = model.transform(test)

    # Métricas
    rmse = evaluator_rmse.evaluate(predictions)
    mae = evaluator_mae.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)

    # Log métricas
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Guardar modelo
    mlflow.spark.log_model(model, "model")

    # ===============================
    # Artefacto 1: Reporte de texto
    # ===============================
    report = f"""
REPORTE DEL MODELO FINAL
=======================
Modelo: Lasso (L1)

Hiperparámetros:
- regParam: {reg_param}
- elasticNetParam: {elastic_param}
- maxIter: {max_iter}

Métricas en test:
- RMSE: ${rmse:,.2f}
- MAE: ${mae:,.2f}
- R²: {r2:.4f}
"""
    mlflow.log_text(report, "model_report.txt")

    # =========================================
    # Artefacto 2: Gráfico Predicho vs Real
    # =========================================

    # Pasar a pandas (sample para no explotar memoria)
    pdf = predictions.select("label", "prediction") \
                     .sample(fraction=0.05, seed=42) \
                     .toPandas()

    plt.figure(figsize=(8, 8))
    plt.scatter(
        pdf["label"],
        pdf["prediction"],
        alpha=0.3
    )

    # Línea ideal y = x
    min_val = min(pdf["label"].min(), pdf["prediction"].min())
    max_val = max(pdf["label"].max(), pdf["prediction"].max())
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

    plt.title("Predicción vs Valor Real\nModelo Lasso – SECOP Contratos")
    plt.xlabel("Valor real del contrato")
    plt.ylabel("Valor predicho del contrato")
    plt.tight_layout()

    # Guardar gráfico temporalmente
    tmp_dir = tempfile.mkdtemp()
    plot_path = os.path.join(tmp_dir, "prediccion_vs_real.png")
    plt.savefig(plot_path)
    plt.close()

    # Log como artefacto
    mlflow.log_artifact(plot_path)


print("\n" + "="*60)
print("MODELO FINAL REGISTRADO EN MLFLOW")
print("="*60)
print(f"RMSE: ${rmse:,.2f}")
print(f"MAE: ${mae:,.2f}")
print(f"R²: {r2:.4f}")
print("-"*60)
print("📌 Todos los experimentos, métricas, modelos y gráficos")
print("📌 pueden revisarse en MLflow UI:")
print("👉 http://localhost:5000")
print("-"*60)
print("Pasos:")
print("1️⃣ Abrir MLflow UI")
print("2️⃣ Entrar al experimento SECOP_Contratos_Prediccion")
print("3️⃣ Seleccionar el run 'best_model_with_artifacts'")
print("4️⃣ Revisar métricas y artefactos (gráficos y reporte)")
print("="*60)



MODELO FINAL REGISTRADO EN MLFLOW
RMSE: $5,200,424,547.61
MAE: $2,757,284,566.63
R²: -0.5838
------------------------------------------------------------
📌 Todos los experimentos, métricas, modelos y gráficos
📌 pueden revisarse en MLflow UI:
👉 http://localhost:5000
------------------------------------------------------------
Pasos:
1️⃣ Abrir MLflow UI
2️⃣ Entrar al experimento SECOP_Contratos_Prediccion
3️⃣ Seleccionar el run 'best_model_with_artifacts'
4️⃣ Revisar métricas y artefactos (gráficos y reporte)


## 7. Reto 5: Preguntas de reflexión

**¿Qué ventajas tiene MLflow sobre guardar métricas en archivos CSV?**

*Respuesta:* Versionado, comparación visual, trazabilidad completa y reproducibilidad.

**¿Cómo implementarías MLflow en un proyecto de equipo?**

*Respuesta:* Implementaría MLflow como un sistema centralizado de tracking de experimentos compartido por todo el equipo, integrándolo desde el inicio del proyecto en los pipelines de entrenamiento. Cada experimento registraría de forma automática los hiperparámetros, métricas y artefactos relevantes, garantizando trazabilidad, reproducibilidad y comparabilidad entre modelos entrenados por distintos integrantes. MLflow se complementaría con git para versionar el código, mientras que la mlflow UI serviría como punto común para analizar resultados y seleccionar el mejor modelo basándose en métricas objetivas.

**¿Qué artefactos adicionales guardarías además del modelo?**

*Respuesta:* Gráficos, pipelines, datasets de validación, reportes de features.

 **¿Cómo automatizarías el registro de experimentos?**
 
*Respuesta:* Integrarlo en pipelines (Airflow, GitHub Actions, Jenkins).


In [23]:
print("\n" + "="*60)
print("RESUMEN MLFLOW TRACKING")
print("="*60)
print("Verifica que hayas completado:")
print("  [✓] Configurado MLflow tracking server")
print("  [✓] Registrado experimento baseline")
print("  [✓] Registrado al menos 3 experimentos adicionales")
print("  [✓] Explorado MLflow UI")
print("  [✓] Comparado métricas entre runs")
print(f"  [ ] Accede a MLflow UI: http://localhost:5000")
print("="*60)


RESUMEN MLFLOW TRACKING
Verifica que hayas completado:
  [✓] Configurado MLflow tracking server
  [✓] Registrado experimento baseline
  [✓] Registrado al menos 3 experimentos adicionales
  [✓] Explorado MLflow UI
  [✓] Comparado métricas entre runs
  [ ] Accede a MLflow UI: http://localhost:5000


In [ ]:
spark.stop()